# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/r6/1fwxrpyn1w937l1v73wpn3jc0000gn/T/ipykernel_67926/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [5]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [6]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Well typically the metadata is the data most people care about, since the data in the RAG application is just an embedding. But beyond that technicality, we often want to give references or sources back to the user so they can confirm and gain confidence in our answers since LLMs are known to hallucinate.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [7]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [8]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

A chunk size of 1 token has lost all context, but a chunk size of 1_000_000+ will contain so much context it has ceased to be useful. Somewhere in between is an optimal chunk size, and it's different for each dataset, but it typically is between 400-800 tokens or about the size of 1 to 2 paragraphs. Essentially, we are looking for each chunk to contain an idea.

For chunk overlap, the larger a chunk size the more duplication we will have in our database so we want them as small as possible to stay efficient. The smaller a chunk size, the more we rely on getting lucky that we have optimal splits. We are trying to prevent cutting off sentences in the middle, or paragraphs in the middle, etc. Situations where a single idea could be split into two seperate chunks and prevent either of them from surfacing during search.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [9]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [10]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [11]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

That the query and the response are either close or far apart in the latent space. Typically it means they are going to have similar word choices, in similar orders. It doesn't really prove anything, so there's many things I could answer here, but this seems like a leading question in which case I'll answer it doesn't prove that the result from the query contains the answer to the user's question.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [12]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [13]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [14]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [15]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs in the provided guideline that may warrant veterinary attention include:

- **Pain or anxiety signs**, especially if new or persistent [Source 1], [Source 4]
- **Changes in grooming**:
  - **Increased grooming** may suggest a skin, allergy, infection, endocrine, or other problem [Source 1]
  - **Reduced grooming** may indicate illness, bladder pain, joint pain, or reduced mobility [Source 2]
- **Changes in appetite, thirst, or urination** (including **polyuria/polydipsia**) [Source 3]
- **Vomiting, vomiting hairballs, or diarrhea**, especially if frequent or ongoing [Source 3]
- **Increased nocturnal activity, vocalization, or changes in normal habits/activity** [Source 3]
- **Fear/stress behaviors** such as cowering, crouching, hiding, freezing, frantic fleeing, flat ears, dilated pupils, or defensive vocalizing like hissing or growling [Source 4]

If you’re seeing any of these, it’s a good idea to contact a veterinarian for an assessment.

Sources:
{'source_label': 'Source 1', 

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [16]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guideline recommends **at least annual veterinary examinations for all cats**, with **more frequent visits based on the cat’s individual needs**. It also notes that **senior cats should be seen at least every 6 months** and more often if they have chronic conditions [Source 4].

For parasite prevention, it says **routine, regular use of broad-spectrum parasite products is likely beneficial for most pet cats**, and prevention should consider the cat’s lifestyle and risk factors. It also recommends treating **housemates at the same time** when needed and limiting access to areas like gardens and children’s sandboxes to reduce parasite exposure [Source 1].

If you want, I can also summarize the other preventive topics mentioned in the guideline, like microchipping, sterilization, and claw care [Source 4].
Question: What symptoms should make me call a veterinarian?
You should call a veterinarian if your cat has changes in appetite

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

We didn't print the retrieved context, but based on the references cited, and content it seems like we have copied some of the phrases from the pdf word for word and they are relevant. So yes.  The last vibe check question is the most interesting, it recieved 4 chunks but still answered correctly.

## 🏗️ Activity: Tune Retrieval Quality with AutoRAG

Eyeballing chunks one query at a time only goes so far. To make tuning evidence-driven, we hand the chunk-size / overlap / top_k sweep to **[AutoRAG](https://github.com/Marker-Inc-Korea/AutoRAG)**, which runs every configuration against the same QA set and produces a `summary.csv` with retrieval F1, recall, precision, NDCG, and MRR per config.

What we'll do:

1. Build a small **gold QA set** (5 questions). For each question we identify the chunk that *actually* contains the answer by anchoring on a distinctive phrase. That chunk's `doc_id` is the retrieval ground-truth.
2. Build one **corpus.parquet per chunk-size config** ((500, 100), (1000, 200), (1500, 300)).
3. Have AutoRAG run a **`semantic_retrieval` node** with **top_k ∈ [2, 4, 6]** for each corpus, all against the same OpenAI embeddings.
4. Read the resulting `summary.csv` files and compare.

**Backend note.** The rest of this notebook uses Qdrant in `:memory:` mode. AutoRAG 0.3.22 ships a Qdrant module too, but it requires a running Qdrant server (Docker or Cloud) — its in-memory fallback is a lexical token-overlap stub, not real OpenAI cosine retrieval. To keep this activity zero-Docker while still measuring **dense** retrieval, we use AutoRAG's `chroma` backend in **persistent** mode (writes embeddings to a folder under `autorag_runs/`). The retrieval semantics — cosine similarity over OpenAI `text-embedding-3-small` — match what Qdrant gave us above. Only the storage layer differs.

Because chunk boundaries differ across configs, the ground-truth `doc_id` can't be reused across corpora. We rebuild the QA set per corpus by re-anchoring on the same distinctive phrase.

In [11]:
# --- AutoRAG setup ---
import shutil
import uuid
from pathlib import Path

import pandas as pd

# Jupyter's IPython kernel runs a top-level asyncio loop so that cell-level
# `await` works. AutoRAG's Evaluator calls `loop.run_until_complete(...)`, which
# is illegal on an already-running loop. nest_asyncio patches the loop to be
# re-entrant. (Outside Jupyter — plain `python script.py` — this is unnecessary.)
import nest_asyncio
nest_asyncio.apply()

AUTORAG_ROOT = Path("autorag_runs")
AUTORAG_ROOT.mkdir(exist_ok=True)
print(f"AutoRAG project dir: {AUTORAG_ROOT.resolve()}")

AutoRAG project dir: /Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/autorag_runs


### Step 1 — Gold QA set

Each entry is `(query, anchor_phrase, gold_answer)`. The anchor is a distinctive substring that appears in **exactly the chunk that contains the answer**. We use it to look up the right `doc_id` after each new chunking config, so the ground-truth tracks the chunk boundaries.

The phrases below were chosen by spot-checking the source PDF for unique strings — `every 6 months` only appears once, `cowering (tense` only appears once, and so on.

In [12]:
QA_SPECS: list[tuple[str, str, str]] = [
    (
        "How often should senior cats be examined?",
        "every 6 months",
        "Senior cats should be seen at least every 6 months, and more frequently for those with chronic conditions.",
    ),
    (
        "What is the resting energy requirement formula for cats?",
        "resting\nenergy requirement",
        "RER (kcal/day) = 30 x body weight in kg + 70.",
    ),
    (
        "What body postures indicate fear or stress in cats?",
        "cowering (tense",
        "A cowering, tense, flattened position with the head lower than the body indicates stress or fear.",
    ),
    (
        "What food-label statement should a cat food carry?",
        "nutritional adequacy",
        "An AAFCO statement of nutritional adequacy.",
    ),
    (
        "What environmental enrichment is recommended for indoor cats?",
        "environmental enrichment",
        "Appropriate environmental enrichment is essential for the mental and physical well-being of indoor cats.",
    ),
]

print(f"{len(QA_SPECS)} QA specs defined.")

5 QA specs defined.


### Step 2 — Helpers: build corpus, build QA, write config

Three small helpers do the per-config work:

- `build_corpus_parquet` chunks the PDF with the given (size, overlap) and writes AutoRAG's required `doc_id / contents / metadata` schema.
- `build_qa_parquet` looks up each anchor phrase in the just-built corpus, takes the matching `doc_id` as `retrieval_gt`, and writes the QA parquet.
- `write_config_yaml` emits an AutoRAG node-line config that runs the `vectordb` (OpenAI embeddings) retrieval module with `top_k ∈ [2, 4, 6]`. AutoRAG auto-explodes list-valued params into separate trial rows.

In [13]:
def build_corpus_parquet(
    pages_in: list,
    chunk_size: int,
    chunk_overlap: int,
    out_path: Path,
) -> pd.DataFrame:
    """Chunk pages with the given config and write AutoRAG's corpus.parquet."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    chunks = splitter.split_documents(pages_in)
    rows = [
        {
            "doc_id": str(uuid.uuid4()),
            "contents": c.page_content,
            "metadata": {
                "page": c.metadata.get("page"),
                "start_index": c.metadata.get("start_index"),
                "last_modified_datetime": pd.Timestamp.now("UTC").isoformat(),
            },
        }
        for c in chunks
    ]
    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False)
    return df


def build_qa_parquet(corpus_df: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    """For each spec, find the chunk containing the anchor phrase and use that
    chunk's doc_id as retrieval ground-truth."""
    rows = []
    for query, anchor, gold in QA_SPECS:
        mask = corpus_df["contents"].str.contains(anchor, case=False, regex=False)
        if not mask.any():
            raise ValueError(f"anchor not found in any chunk: {anchor!r} (query={query!r})")
        gt_id = corpus_df.loc[mask, "doc_id"].iloc[0]
        rows.append(
            {
                "qid": str(uuid.uuid4()),
                "query": query,
                "retrieval_gt": [[gt_id]],   # AutoRAG: 2-D — outer = AND-groups
                "generation_gt": [gold],
            }
        )
    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False)
    return df


def write_config_yaml(out_path: Path, top_k_values: list[int]) -> None:
    """AutoRAG 0.3.22 schema: top-level `vectordb:` block names a backend, then
    a `semantic_retrieval` node references it. Each unique chroma collection
    name avoids the previous run's embeddings being reused.
    """
    out_path.write_text(
        f"""vectordb:
  - name: chroma_local
    db_type: chroma
    client_type: persistent
    embedding_model: openai
    collection_name: cat_health
    path: ${{PROJECT_DIR}}/resources/chroma
    embedding_batch: 64
node_lines:
  - node_line_name: retrieve_only
    nodes:
      - node_type: semantic_retrieval
        strategy:
          metrics: [retrieval_f1, retrieval_recall, retrieval_precision, retrieval_ndcg, retrieval_mrr]
        top_k: {top_k_values}
        modules:
          - module_type: vectordb
            vectordb: chroma_local
""",
        encoding="utf-8",
    )

### Step 3 — Run AutoRAG once per chunk-size config

For each of `(500/100)`, `(1000/200)`, `(1500/300)`:

1. Wipe and create a fresh project dir under `autorag_runs/`.
2. Build `corpus.parquet`, `qa.parquet`, and `config.yaml`.
3. `Evaluator.start_trial()` runs the `vectordb` retrieval module three times (once per `top_k`) and writes a `summary.csv`.
4. We accumulate the summary rows, tagging each with the chunk config that produced them.

This re-embeds the corpus once per config, so it makes 3 × N OpenAI embedding calls plus 3 × 3 query-embedding calls. With our 5 questions and ~70-200 chunks per config, total embedding cost is small (well under a cent on `text-embedding-3-small`).

In [14]:
# Import AutoRAG only after the corpus/qa helpers are defined.
from autorag.evaluator import Evaluator

CHUNK_CONFIGS = [(500, 100), (1000, 200), (1500, 300)]
TOP_K_VALUES = [2, 4, 6]

all_rows: list[pd.DataFrame] = []

for cs, ov in CHUNK_CONFIGS:
    project_dir = AUTORAG_ROOT / f"chunk_{cs}_{ov}"
    if project_dir.exists():
        shutil.rmtree(project_dir)
    data_dir = project_dir / "data"
    data_dir.mkdir(parents=True)

    print(f"\n=== chunk_size={cs}, overlap={ov} ===")
    corpus_df = build_corpus_parquet(pages, cs, ov, data_dir / "corpus.parquet")
    qa_df = build_qa_parquet(corpus_df, data_dir / "qa.parquet")
    cfg_path = project_dir / "config.yaml"
    write_config_yaml(cfg_path, TOP_K_VALUES)
    print(f"  corpus chunks: {len(corpus_df)}, qa pairs: {len(qa_df)}")

    Evaluator(
        qa_data_path=str(data_dir / "qa.parquet"),
        corpus_data_path=str(data_dir / "corpus.parquet"),
        project_dir=str(project_dir),
    ).start_trial(str(cfg_path))

    trial_dirs = sorted(p for p in project_dir.iterdir() if p.is_dir() and p.name.isdigit())
    summary = pd.read_csv(trial_dirs[-1] / "retrieve_only" / "semantic_retrieval" / "summary.csv")
    summary["chunk_size"] = cs
    summary["chunk_overlap"] = ov
    all_rows.append(summary)

sweep = pd.concat(all_rows, ignore_index=True)
print("\n=== combined sweep ===")
print(sweep.to_string(index=False))


=== chunk_size=500, overlap=100 ===
  corpus chunks: 263, qa pairs: 5


[06/07/26 22:23:30] INFO     [evaluator.py:130] >>                                                 ]8;id=15932126;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py\evaluator.py]8;;\:]8;id=15932127;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py#130\130]8;;\
                                             _        _____            _____                                       
                                  /\        | |      |  __ \     /\   / ____|                                      
                                 /  \  _   _| |_ ___ | |__) |   /  \ | |  __                                       
                                / /\ \| | | | __/ _ \|  _  /   / /\ \| | |_ |                                      
                               / ____ \ |_| | || (_) | | \ \  / ____ \ |__| |                                      
                              /_/    \_\__,_|\__\___/|_|  \_\/_/    \_\_____|                                      
                                                                                                                   
                                                                                                                   

                    INFO     [evaluator.py:131] >> Start Validation input data and config YAML     ]8;id=15932132;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py\evaluator.py]8;;\:]8;id=15932133;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py#131\131]8;;\
                             file first. If you want to skip this, put the --skip_validation flag                  
                             or `skip_validation` at the start_trial function.                                     

[06/07/26 22:23:31] INFO     [_client.py:1740] >> HTTP Request: POST                                ]8;id=15932140;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15932141;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"                                

                    INFO     [evaluator.py:204] >> Running node line retrieve_only...              ]8;id=15932147;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py\evaluator.py]8;;\:]8;id=15932148;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py#204\204]8;;\

...

                    INFO     [_client.py:1740] >> HTTP Request: POST                                ]8;id=15933218;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15933219;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"                                

                    INFO     [_client.py:1740] >> HTTP Request: POST                                ]8;id=15933224;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15933225;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"                                

                    INFO     [base.py:28] >> Deleting retrieval node - VectorDB module...                ]8;id=15933230;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/nodes/retrieval/base.py\base.py]8;;\:]8;id=15933231;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/nodes/retrieval/base.py#28\28]8;;\

                    INFO     [evaluator.py:219] >> Evaluation complete.                            ]8;id=15933236;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py\evaluator.py]8;;\:]8;id=15933237;file:///Users/msharp/projects/aiec1/01_Dense_Vector_Retrieval/.venv/lib/python3.12/site-packages/autorag/evaluator.py#219\219]8;;\


=== combined sweep ===
 filename module_name                            module_params  execution_time  retrieval_f1  retrieval_recall  retrieval_precision  retrieval_ndcg  retrieval_mrr  is_best  chunk_size  chunk_overlap
0.parquet    VectorDB {'top_k': 2, 'vectordb': 'chroma_local'}        0.060160      0.666667               1.0             0.500000        1.000000           1.00     True         500            100
1.parquet    VectorDB {'top_k': 4, 'vectordb': 'chroma_local'}        0.094695      0.400000               1.0             0.250000        1.000000           1.00    False         500            100
2.parquet    VectorDB {'top_k': 6, 'vectordb': 'chroma_local'}        0.104023      0.285714               1.0             0.166667        1.000000           1.00    False         500            100
0.parquet    VectorDB {'top_k': 2, 'vectordb': 'chroma_local'}        0.084996      0.533333               0.8             0.400000        0.800000           0.80    False        1

### Step 4 — Pick the winner

Slim the sweep down to (chunk_size, overlap, top_k) → metrics, sort by `retrieval_ndcg`, and print the top configurations. NDCG balances "did we retrieve a relevant chunk" with "is it ranked near the top," which is the right proxy for downstream RAG quality. We also print the recall column to see whether any config simply fails to retrieve the gold chunk.

In [15]:
def parse_top_k(s: str) -> int:
    # module_params is a python-repr dict like
    # "{'top_k': 4, 'vectordb': 'chroma_local'}"; eval is fine, our own data.
    return int(eval(s)["top_k"])


leaderboard = sweep.copy()
leaderboard["top_k"] = leaderboard["module_params"].apply(parse_top_k)
leaderboard = leaderboard[
    [
        "chunk_size",
        "chunk_overlap",
        "top_k",
        "retrieval_recall",
        "retrieval_precision",
        "retrieval_f1",
        "retrieval_ndcg",
        "retrieval_mrr",
    ]
].sort_values("retrieval_ndcg", ascending=False).reset_index(drop=True)

print(leaderboard.to_string(index=False))
print(f"\nBest config (by NDCG): {leaderboard.iloc[0].to_dict()}")

 chunk_size  chunk_overlap  top_k  retrieval_recall  retrieval_precision  retrieval_f1  retrieval_ndcg  retrieval_mrr
        500            100      2               1.0             0.500000      0.666667        1.000000           1.00
        500            100      4               1.0             0.250000      0.400000        1.000000           1.00
        500            100      6               1.0             0.166667      0.285714        1.000000           1.00
       1000            200      4               1.0             0.250000      0.400000        0.886135           0.85
       1000            200      6               1.0             0.166667      0.285714        0.886135           0.85
       1000            200      2               0.8             0.400000      0.533333        0.800000           0.80
       1500            300      2               0.8             0.400000      0.533333        0.800000           0.80
       1500            300      4               0.8     

### 🏗️ Activity Notes

**Setting changed:** chunk_size, chunk_overlap, and `top_k`, swept jointly.

**Sweep grid:** chunk_size ∈ {500, 1000, 1500} × chunk_overlap ∈ {100, 200, 300} (paired, 1:5 ratio) × top_k ∈ {2, 4, 6}, evaluated against a 5-question gold QA set with one ground-truth chunk per question (anchored on a distinctive phrase).

**Leaderboard (sorted by retrieval_ndcg):**

| chunk_size | overlap | top_k | recall | precision | F1   | NDCG | MRR  |
|-----------:|--------:|------:|-------:|----------:|-----:|-----:|-----:|
| **500**    | **100** | **2** | 1.00   | 0.50      | 0.67 | 1.00 | 1.00 |
| 500        | 100     | 4     | 1.00   | 0.25      | 0.40 | 1.00 | 1.00 |
| 500        | 100     | 6     | 1.00   | 0.17      | 0.29 | 1.00 | 1.00 |
| 1000       | 200     | 4     | 1.00   | 0.25      | 0.40 | 0.89 | 0.85 |
| 1000       | 200     | 6     | 1.00   | 0.17      | 0.29 | 0.89 | 0.85 |
| 1000       | 200     | 2     | 0.80   | 0.40      | 0.53 | 0.80 | 0.80 |
| 1500       | 300     | 2     | 0.80   | 0.40      | 0.53 | 0.80 | 0.80 |
| 1500       | 300     | 4     | 0.80   | 0.20      | 0.32 | 0.80 | 0.80 |
| 1500       | 300     | 6     | 0.80   | 0.13      | 0.23 | 0.80 | 0.80 |

**Headline:** all three `chunk_size=500, overlap=100` runs score **NDCG 1.00 + MRR 1.00** — the gold chunk is always ranked #1. The 500/100/`top_k=2` row wins on F1 because precision is highest there.

**Before:** notebook default of `chunk_size=1000, chunk_overlap=200, k=4` → recall 1.00, NDCG 0.89, MRR 0.85.

**After:** `chunk_size=500, chunk_overlap=100, top_k=2` → recall 1.00, NDCG 1.00 (+12%), MRR 1.00 (+18%), F1 0.67 (+67% vs the default's 0.40). Smaller chunks make the answer-bearing chunk topically pure, so it ranks #1 reliably; smaller `top_k` then captures the precision win without losing recall.

**Did retrieval improve?** Yes, decisively. Three observations from the table:
1. **`top_k` doesn't change retrieval ranking quality, only precision.** Recall, NDCG, and MRR are flat across `top_k` for any fixed (chunk_size, overlap). That makes sense — the retriever returns the same top-K *ordering*; cranking K just dilutes precision by adding lower-ranked passages.
2. **Smaller chunks dominate.** 500-char chunks land the gold chunk at rank 1 every time (NDCG=MRR=1.0). 1000-char chunks miss rank 1 on at least one question, so MRR drops to ~0.85. 1500-char chunks dilute the embedding enough that one question's gold chunk never appears at all (recall=0.80 across all top_k).
3. **Overlap didn't earn its keep here.** A larger overlap was supposed to protect against unlucky boundary splits, but at 1500/300 we still lost a question's gold chunk. The hypothesis is that on this PDF the relevant phrase-cluster fits inside a 500-char window, so giving the embedder more text only adds neighboring topics that pull the cosine direction off-target.

**Caveats:**
- 5 QA pairs is a small N — recall jumps in 0.20 increments, so anything that swings one question swings the whole metric. A more rigorous sweep would use ≥30 questions or AutoRAG's own QA-generation (`autorag.data.qa.*`) to scale the gold set.
- The `retrieval_gt` is one-chunk-per-question, anchored on a specific phrase. Questions whose answer genuinely spans multiple chunks aren't measured here.
- We swept chunk_size and overlap together (1:5 ratio). Decoupling them — e.g., 500/0 vs 500/100 vs 500/200 — would tell us which one is doing the work.